In [13]:
import pandas as pd

data = pd.read_csv("train.csv")

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [14]:
# Drop useless columns
data = data.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [16]:
# Split data into X, and y
X = data.drop(columns=["Survived"])
y = data["Survived"]

X.info()
print("\n")
y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    object 
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Fare      891 non-null    float64
 6   Embarked  889 non-null    object 
dtypes: float64(2), int64(3), object(2)
memory usage: 48.9+ KB


<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Survived
Non-Null Count  Dtype
--------------  -----
891 non-null    int64
dtypes: int64(1)
memory usage: 7.1 KB


In [17]:
# Split X and y into train, cv, test
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state = 42)
X_cv, X_test, y_cv, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state = 42)

# Check
print(X_train.shape)
print(X_cv.shape)
print(X_test.shape)
print(y_train.shape, y_cv.shape, y_train.shape, sep = "\n")

(534, 7)
(178, 7)
(179, 7)
(534,)
(178,)
(534,)


In [18]:
# Find missing values
print(f"X_train:\n{X_train.isnull().any()}\n")
print(f"X_cv:\n{X_cv.isnull().any()}\n")
print(f"X_test:\n{X_test.isnull().any()}\n")

X_train:
Pclass      False
Sex         False
Age          True
SibSp       False
Parch       False
Fare        False
Embarked    False
dtype: bool

X_cv:
Pclass      False
Sex         False
Age          True
SibSp       False
Parch       False
Fare        False
Embarked     True
dtype: bool

X_test:
Pclass      False
Sex         False
Age          True
SibSp       False
Parch       False
Fare        False
Embarked     True
dtype: bool



In [19]:
# Fix missing values under X_train: 1) Age
age_median = X_train["Age"].median()

X_train["Age"] = X_train["Age"].fillna(age_median)
X_cv["Age"] = X_cv["Age"].fillna(age_median)
X_test["Age"] = X_test["Age"].fillna(age_median)

# Check
print(f"X_train:{X_train["Age"].isnull().any()}")
print(f"X_cv:{X_cv["Age"].isnull().any()}")
print(f"X_test:{X_test["Age"].isnull().any()}")

X_train:False
X_cv:False
X_test:False


In [20]:
# Fix missing values under X_train: 2) Embarked
# Embarked has S C Q, so we will use mode() -> most frequent elem

emb_freq = X_train["Embarked"].mode()[0]

X_cv["Embarked"] = X_cv["Embarked"].fillna(emb_freq)
X_test["Embarked"] = X_test["Embarked"].fillna(emb_freq)


# Check
print(f"X_train:{X_train["Embarked"].isnull().any()}")
print(f"X_cv:{X_cv["Embarked"].isnull().any()}")
print(f"X_test:{X_test["Embarked"].isnull().any()}")

X_train:False
X_cv:False
X_test:False


In [21]:
# Embarked having values like S C Q , we need to change them into 0 1 2
X_train = pd.get_dummies(X_train, columns=["Embarked"])
X_cv = pd.get_dummies(X_cv, columns=["Embarked"])
X_test = pd.get_dummies(X_test, columns=["Embarked"])

In [22]:
# Check
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
570,2,male,62.0,0,0,10.5000,False,False,True
787,3,male,8.0,4,1,29.1250,False,True,False
74,3,male,32.0,0,0,56.4958,False,False,True
113,3,female,20.0,1,0,9.8250,False,False,True
635,2,female,28.0,0,0,13.0000,False,False,True


In [23]:
# Change colums Sex male and female into 0 and 1
X_train["Sex"] = X_train["Sex"].map({"male": 0, "female": 1})
X_cv["Sex"] = X_cv["Sex"].map({"male": 0, "female": 1})
X_test["Sex"] = X_test["Sex"].map({"male": 0, "female": 1})

In [24]:
# Check
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
570,2,0,62.0,0,0,10.5000,False,False,True
787,3,0,8.0,4,1,29.1250,False,True,False
74,3,0,32.0,0,0,56.4958,False,False,True
113,3,1,20.0,1,0,9.8250,False,False,True
635,2,1,28.0,0,0,13.0000,False,False,True


In [25]:
# Imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [26]:
# Model: Baseline
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [27]:
y_trpred = rf_clf.predict(X_train)
y_cvpred = rf_clf.predict(X_cv)
y_tspred = rf_clf.predict(X_test)

acc_tr = accuracy_score(y_train, y_trpred)
acc_cv = accuracy_score(y_cv, y_cvpred)
acc_tst = accuracy_score(y_test, y_tspred)

print(acc_tr, acc_cv, acc_tst, sep="\n")

0.9831460674157303
0.7640449438202247
0.7988826815642458


In [30]:
# n_estimator tuning
number_estim = [i*20 for i in range(1, 15)]

cv_hist = []

for i in number_estim:
  model = RandomForestClassifier(n_estimators= i, random_state=42)
  model.fit(X_train, y_train)

  y_trpred = model.predict(X_train)
  y_cvpred = model.predict(X_cv)
  y_tspred = model.predict(X_test)

  acc_tr = accuracy_score(y_train, y_trpred)
  acc_cv = accuracy_score(y_cv, y_cvpred)
  acc_tst = accuracy_score(y_test, y_tspred)

  cv_hist.append(acc_cv)

  print(f"train: {acc_tr}, cv: {acc_cv}, test: {acc_tst}")



train: 0.9812734082397003, cv: 0.7640449438202247, test: 0.7877094972067039
train: 0.9831460674157303, cv: 0.7640449438202247, test: 0.7932960893854749
train: 0.9831460674157303, cv: 0.7640449438202247, test: 0.7932960893854749
train: 0.9831460674157303, cv: 0.7696629213483146, test: 0.7932960893854749
train: 0.9831460674157303, cv: 0.7640449438202247, test: 0.7988826815642458
train: 0.9831460674157303, cv: 0.7752808988764045, test: 0.7988826815642458
train: 0.9831460674157303, cv: 0.7696629213483146, test: 0.7932960893854749
train: 0.9831460674157303, cv: 0.7584269662921348, test: 0.7988826815642458
train: 0.9831460674157303, cv: 0.7696629213483146, test: 0.7988826815642458
train: 0.9831460674157303, cv: 0.7696629213483146, test: 0.7988826815642458
train: 0.9831460674157303, cv: 0.7696629213483146, test: 0.7988826815642458
train: 0.9831460674157303, cv: 0.7696629213483146, test: 0.7988826815642458
train: 0.9831460674157303, cv: 0.7640449438202247, test: 0.7988826815642458
train: 0.983

In [33]:
print(max(cv_hist))
best_estim = number_estim[cv_hist.index(max(cv_hist))]
print(best_estim)

0.7752808988764045
120


In [34]:
# max depth
cv_hist = []

for i in range(2, 21):
  model = RandomForestClassifier(n_estimators= best_estim,max_depth= i, random_state=42)
  model.fit(X_train, y_train)

  y_trpred = model.predict(X_train)
  y_cvpred = model.predict(X_cv)
  y_tspred = model.predict(X_test)

  acc_tr = accuracy_score(y_train, y_trpred)
  acc_cv = accuracy_score(y_cv, y_cvpred)
  acc_tst = accuracy_score(y_test, y_tspred)

  cv_hist.append(acc_cv)

  print(f"train: {acc_tr}, cv: {acc_cv}, test: {acc_tst}")

train: 0.8164794007490637, cv: 0.7808988764044944, test: 0.776536312849162
train: 0.8520599250936329, cv: 0.8033707865168539, test: 0.8100558659217877
train: 0.8632958801498127, cv: 0.8089887640449438, test: 0.8212290502793296
train: 0.8782771535580525, cv: 0.8033707865168539, test: 0.8212290502793296
train: 0.897003745318352, cv: 0.8089887640449438, test: 0.8100558659217877
train: 0.9026217228464419, cv: 0.8202247191011236, test: 0.8156424581005587
train: 0.9213483146067416, cv: 0.8089887640449438, test: 0.7932960893854749
train: 0.9400749063670412, cv: 0.7865168539325843, test: 0.8156424581005587
train: 0.9569288389513109, cv: 0.7921348314606742, test: 0.8100558659217877
train: 0.9662921348314607, cv: 0.7808988764044944, test: 0.8100558659217877
train: 0.9719101123595506, cv: 0.7808988764044944, test: 0.8100558659217877
train: 0.9775280898876404, cv: 0.7752808988764045, test: 0.8044692737430168
train: 0.9812734082397003, cv: 0.7696629213483146, test: 0.8044692737430168
train: 0.98314

In [37]:
print(max(cv_hist))
best_depth = cv_hist.index(max(cv_hist)) + 2

print(best_depth)

0.8202247191011236
7


In [52]:
# n_estimator tuning
min_smpl = [i*5 for i in range(1, 100)]

cv_hist = []

for i in min_smpl:
  model = RandomForestClassifier(n_estimators= best_estim,max_depth=best_depth, min_samples_split=i,random_state=42)
  model.fit(X_train, y_train)

  y_trpred = model.predict(X_train)
  y_cvpred = model.predict(X_cv)
  y_tspred = model.predict(X_test)

  acc_tr = accuracy_score(y_train, y_trpred)
  acc_cv = accuracy_score(y_cv, y_cvpred)
  acc_tst = accuracy_score(y_test, y_tspred)

  cv_hist.append(acc_cv)

  print(f"train: {acc_tr}, cv: {acc_cv}, test: {acc_tst}")


train: 0.897003745318352, cv: 0.8033707865168539, test: 0.8100558659217877
train: 0.8801498127340824, cv: 0.8089887640449438, test: 0.8212290502793296
train: 0.8764044943820225, cv: 0.8033707865168539, test: 0.8156424581005587
train: 0.8726591760299626, cv: 0.8033707865168539, test: 0.8156424581005587
train: 0.8745318352059925, cv: 0.8089887640449438, test: 0.8156424581005587
train: 0.8632958801498127, cv: 0.8089887640449438, test: 0.8156424581005587
train: 0.8651685393258427, cv: 0.8146067415730337, test: 0.8044692737430168
train: 0.8558052434456929, cv: 0.8089887640449438, test: 0.8156424581005587
train: 0.8539325842696629, cv: 0.8033707865168539, test: 0.8212290502793296
train: 0.846441947565543, cv: 0.8089887640449438, test: 0.8100558659217877
train: 0.8576779026217228, cv: 0.8146067415730337, test: 0.8044692737430168
train: 0.8576779026217228, cv: 0.8089887640449438, test: 0.8100558659217877
train: 0.8558052434456929, cv: 0.8089887640449438, test: 0.7988826815642458
train: 0.85393

In [53]:
print(max(cv_hist))
best_min_smpl = min_smpl[cv_hist.index(max(cv_hist))]
print(best_min_smpl)

0.8146067415730337
35


In [56]:
# min_samples_leaf tuning
min_smpl_leaf = [i for i in range(1, 11)]

cv_hist = []

for i in min_smpl_leaf:
    model = RandomForestClassifier(
        n_estimators=best_estim,
        max_depth=best_depth,
        min_samples_split=best_min_smpl,
        min_samples_leaf=i,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_trpred = model.predict(X_train)
    y_cvpred = model.predict(X_cv)
    y_tspred = model.predict(X_test)

    acc_tr = accuracy_score(y_train, y_trpred)
    acc_cv = accuracy_score(y_cv, y_cvpred)
    acc_tst = accuracy_score(y_test, y_tspred)

    cv_hist.append(acc_cv)

    print(f"train: {acc_tr}, cv: {acc_cv}, test: {acc_tst}")

print(max(cv_hist))

best_min_leaf = min_smpl_leaf[cv_hist.index(max(cv_hist))]
print(best_min_leaf)

train: 0.8651685393258427, cv: 0.8146067415730337, test: 0.8044692737430168
train: 0.8632958801498127, cv: 0.8089887640449438, test: 0.7988826815642458
train: 0.8614232209737828, cv: 0.8033707865168539, test: 0.8044692737430168
train: 0.8614232209737828, cv: 0.797752808988764, test: 0.8100558659217877
train: 0.8576779026217228, cv: 0.7921348314606742, test: 0.8100558659217877
train: 0.850187265917603, cv: 0.797752808988764, test: 0.8156424581005587
train: 0.846441947565543, cv: 0.8089887640449438, test: 0.8156424581005587
train: 0.8445692883895131, cv: 0.8089887640449438, test: 0.8156424581005587
train: 0.8408239700374532, cv: 0.8089887640449438, test: 0.8156424581005587
train: 0.8426966292134831, cv: 0.8033707865168539, test: 0.8156424581005587
0.8146067415730337
1


In [57]:
inputs = ["sqrt", "log2", None]

cv_hist = []

for i in inputs:
  model = RandomForestClassifier(n_estimators= best_estim, max_depth=best_depth, min_samples_split=best_min_smpl, min_samples_leaf=best_min_leaf, max_features=i, random_state=42)
  model.fit(X_train, y_train)

  y_trpred = model.predict(X_train)
  y_cvpred = model.predict(X_cv)
  y_tspred = model.predict(X_test)

  acc_tr = accuracy_score(y_train, y_trpred)
  acc_cv = accuracy_score(y_cv, y_cvpred)
  acc_tst = accuracy_score(y_test, y_tspred)

  cv_hist.append(acc_cv)

  print(f"train: {acc_tr}, cv: {acc_cv}, test: {acc_tst}")

train: 0.8651685393258427, cv: 0.8146067415730337, test: 0.8044692737430168
train: 0.8651685393258427, cv: 0.8146067415730337, test: 0.8044692737430168
train: 0.8614232209737828, cv: 0.7808988764044944, test: 0.7988826815642458


In [59]:
print(f"Bestmax: {max(cv_hist)}")

Bestmax: 0.8146067415730337
